# Rubout watermark detector, RESUME / settling phase
Continues from your saved `rubout_best.pt` with a low learning rate that decays to zero (about 100 min).
Trains a bigger U-Net to find watermarks (people/face/body photos, hard overlays) and exports `detector.onnx` + `detector.json`.

**How to run:** Colab: Runtime > Change runtime type > GPU (T4), then Runtime > Run all.
Kaggle: Settings > Accelerator GPU, Internet ON, then Run All.
It trains until validation IoU reaches `TARGET_IOU` or `MAX_MINUTES` is used up, whichever comes first, then downloads two files. Send those two files back to Claude.

In [ ]:
TARGET_IOU = 0.80     # stop early when validation IoU reaches this
MAX_MINUTES = 100     # settling phase: lr 0.0004 decays to ~0
BATCH = 24
CROP = 256
WIDTH = 24            # model width (16 = fast on phones; 24 = more accurate, slower)

In [ ]:
import os, sys, subprocess
def sh(c): print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-1500:])
sh('pip -q install onnx onnxscript onnxruntime scikit-image scikit-learn pillow 2>&1 | tail -2')
sh('apt-get -qq update; apt-get -qq install -y fontconfig fonts-dejavu fonts-liberation fonts-freefont-ttf fonts-roboto fonts-open-sans fonts-lato fonts-hack fonts-inconsolata 2>&1 | tail -2')
sh('git -C rubout pull 2>&1 | tail -1' if os.path.exists('rubout') else 'git clone --depth 1 https://github.com/body1990/rubout rubout 2>&1 | tail -1')
sh('python rubout/ci/train/prep_people.py 2>&1 | tail -8')

In [ ]:
import numpy as np, random, time, math, json, torch, torch.nn as nn, torch.nn.functional as F
os.environ['PHOTOS'] = os.path.abspath('rubout/ci/photos') + '/'
os.environ['EXTRA'] = '/tmp/people'
sys.path.insert(0, os.path.abspath('rubout/ci/train'))
import datagen
print('backgrounds:', len(datagen.PH), 'people photos:', len(datagen.PEOPLE), 'fonts:', len(datagen.FONTS))
assert len(datagen.PEOPLE) > 1000, 'Face photos did not download (see the lfw lines above). Re-run the previous cell, or tell Claude.'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
assert dev == 'cuda', 'Turn on a GPU runtime first (Runtime > Change runtime type).'

class Stream(torch.utils.data.IterableDataset):
    def __iter__(self):
        wi = torch.utils.data.get_worker_info(); wid = wi.id if wi else 0
        seed = int(time.time() * 1000) % 100000 * 1000 + wid * 7919; n = 0
        while True:
            rng = random.Random(seed + n); np.random.seed((seed + n) % (2**31)); n += 1
            X, Y = datagen.sample(rng, datagen.PH, CROP)
            yield torch.from_numpy(X.copy()).permute(2, 0, 1), torch.from_numpy(Y.astype(np.float32))[None]
nw = min(8, os.cpu_count() or 2)
loader = torch.utils.data.DataLoader(Stream(), batch_size=BATCH, num_workers=nw, prefetch_factor=4, persistent_workers=True)

# fixed validation set
rv = random.Random(123456789); np.random.seed(5)
vx, vy = [], []
for i in range(600):
    X, Y = datagen.sample(rv, datagen.PH, CROP); vx.append(X); vy.append(Y)
VX = torch.from_numpy(np.stack(vx)).permute(0, 3, 1, 2); VY = torch.from_numpy(np.stack(vy).astype(np.float32))[:, None]
print('val', VX.shape, 'mask fraction', VY.mean().item())

In [ ]:
def cb(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
class UNet(nn.Module):
    def __init__(s, w=16):
        super().__init__(); c = [w, 2*w, 4*w, 8*w, 12*w]
        s.e1 = cb(3, c[0]); s.e2 = cb(c[0], c[1]); s.e3 = cb(c[1], c[2]); s.e4 = cb(c[2], c[3]); s.b = cb(c[3], c[4])
        s.d4 = cb(c[4] + c[3], c[3]); s.d3 = cb(c[3] + c[2], c[2]); s.d2 = cb(c[2] + c[1], c[1]); s.d1 = cb(c[1] + c[0], c[0])
        s.out = nn.Conv2d(c[0], 1, 1)
    def forward(s, x):               # x: float, already normalised
        e1 = s.e1(x); e2 = s.e2(F.max_pool2d(e1, 2)); e3 = s.e3(F.max_pool2d(e2, 2)); e4 = s.e4(F.max_pool2d(e3, 2))
        b = s.b(F.max_pool2d(e4, 2)); up = lambda t: F.interpolate(t, scale_factor=2, mode='nearest')
        d4 = s.d4(torch.cat([up(b), e4], 1)); d3 = s.d3(torch.cat([up(d4), e3], 1))
        d2 = s.d2(torch.cat([up(d3), e2], 1)); d1 = s.d1(torch.cat([up(d2), e1], 1))
        return s.out(d1)
def prep(xu8): return xu8.float() / 255.0 - 0.5
model = UNet(WIDTH).to(dev).to(memory_format=torch.channels_last)
print('params', sum(p.numel() for p in model.parameters()))
from google.colab import files
print('Choose rubout_best.pt (the file you downloaded earlier)')
_up = files.upload(); model.load_state_dict(torch.load(list(_up)[0], map_location=dev)); print('resumed from', list(_up)[0])
def loss_fn(lg, y):
    bce = F.binary_cross_entropy_with_logits(lg, y, pos_weight=torch.tensor(1.5, device=lg.device))
    p = torch.sigmoid(lg); inter = (p * y).sum((1, 2, 3)); dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return bce + 0.5 * dice.mean()
@torch.no_grad()
def val_probs():
    model.eval(); out = []
    for i in range(0, len(VX), 50):
        with torch.autocast('cuda', dtype=torch.float16): out.append(torch.sigmoid(model(prep(VX[i:i+50].to(dev)))).float().cpu())
    model.train(); return torch.cat(out)
def iou_at(P, thr):
    p = P > thr; y = VY > 0.5; tp = (p & y).sum().item(); fp = (p & ~y).sum().item(); fn = (~p & y).sum().item()
    return tp / max(1, tp + fp + fn), tp / max(1, tp + fn), tp / max(1, tp + fp)
def best_thr(P):
    return max(((iou_at(P, t)[0], t) for t in np.arange(0.2, 0.96, 0.05)))

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4); scaler = torch.cuda.amp.GradScaler()
t0 = time.time(); step = 0; best = (-1, None); budget = MAX_MINUTES * 60; lr0 = 4e-4; done = False
model.train()
while not done:
    for x, y in loader:
        el = time.time() - t0; frac = min(1.0, el / budget)
        # random 90-degree rotations and flips on the batch
        k = random.randint(0, 3)
        if k: x = torch.rot90(x, k, (2, 3)); y = torch.rot90(y, k, (2, 3))
        if random.random() < 0.5: x = x.flip(3); y = y.flip(3)
        lr = lr0 * min(1, (step + 1) / 200) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
        for g in opt.param_groups: g['lr'] = lr
        x = x.to(dev, non_blocking=True).contiguous(memory_format=torch.channels_last); y = y.to(dev, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16): lg = model(prep(x))
        loss = loss_fn(lg.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); step += 1
        if step % 400 == 0:
            P = val_probs(); iou, thr = best_thr(P); _, rec, prec = iou_at(P, thr)
            print(f'step {step} loss {loss.item():.3f} lr {lr:.5f} val IoU {iou:.3f} @thr {thr:.2f} (recall {rec:.2f} precision {prec:.2f}) {el/60:.1f} min', flush=True)
            if iou > best[0]: best = (iou, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            if iou >= TARGET_IOU: print('TARGET REACHED'); done = True; break
        if frac >= 1: done = True; break
model.load_state_dict(best[1]); model.to(dev)
P = val_probs(); iou, thr = best_thr(P); print('BEST val IoU', round(iou, 3), 'at threshold', round(float(thr), 2))

In [ ]:
# calibrate three sensitivity levels from the validation set
curve = [(float(t), *iou_at(P, t)) for t in np.arange(0.1, 0.96, 0.025)]
bal = max(curve, key=lambda r: r[1])[0]
gentle = min([r[0] for r in curve if r[3] >= 0.85 and r[0] >= bal] or [min(0.95, bal + 0.15)])
strong = max([r[0] for r in curve if r[2] >= 0.85 and r[0] <= bal] or [max(0.1, bal - 0.15)])
thr3 = [round(gentle, 3), round(bal, 3), round(strong, 3)]
print('thresholds (gentle, balanced, strong):', thr3)

class Export(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x): return torch.sigmoid(s.m(x.float() / 255.0 - 0.5))
model = model.to(memory_format=torch.contiguous_format).float().cpu().eval(); ex = Export(model).eval()
dummy = torch.randint(0, 255, (1, 3, 256, 320), dtype=torch.uint8)
kw = dict(input_names=['image'], output_names=['prob'], opset_version=17, dynamic_axes={'image': {2: 'h', 3: 'w'}, 'prob': {2: 'h', 3: 'w'}})
try: torch.onnx.export(ex, dummy, 'detector.onnx', dynamo=False, **kw)
except TypeError: torch.onnx.export(ex, dummy, 'detector.onnx', **kw)
import onnxruntime as ort
so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
s = ort.InferenceSession('detector.onnx', so, providers=['CPUExecutionProvider'])
for hw in [(256, 320), (384, 512)]:
    t = torch.randint(0, 255, (1, 3) + hw, dtype=torch.uint8)
    with torch.no_grad(): b = ex(t).numpy()
    a = s.run(None, {'image': t.numpy()})[0]
    print(hw, 'onnx vs torch: max diff', float(np.abs(a - b).max()), 'mean diff', float(np.abs(a - b).mean()), 'torch mean', float(b.mean()), 'onnx mean', float(a.mean()))
json.dump({'thr': thr3, 'iou': round(iou, 4), 'multiple': 16, 'input': 'uint8 NCHW RGB, H and W multiples of 16', 'width': WIDTH}, open('detector.json', 'w'))
print(os.path.getsize('detector.onnx') / 1e6, 'MB')

In [ ]:
try:
    from google.colab import files
    files.download('detector.onnx'); files.download('detector.json')
except Exception as e:
    print('Files are in the working folder (Kaggle: see the Output tab):', os.listdir('.'))